# Imports and initialization

In [1]:
import re

import logging
import os
from pathlib import Path
import pandas as pd

import spacy
from spacy import displacy
from spacy.tokens.doc import Doc

# defining paths for notebook
current_dir = Path(globals()["_dh"][0])

csv_folder = os.path.join(current_dir, r"csv")
train_csv = "train.csv"

# logging configuration
logging.basicConfig(level=logging.DEBUG)

## Reading csv

In [2]:
# We are not using title, but we'd like to, add it here
used_columns = ["id", "text", "rating"]

df = pd.read_csv(os.path.join(csv_folder, train_csv), usecols=used_columns)

text_test = df["text"][10]

df.head()

,id,text,rating
0,17578,"Por incrível que pareça, para uma bebida desti...",5
1,18658,"O readset pode até ser bom, mais tem outros fo...",1
2,28477,"Foi difícil terminar esse livro , demorou mese...",2
3,43638,"A bola é boa divertida, mas não é nem um pouco...",2
4,26130,Comprei errado! Não tenho leitor de e-books. Q...,1


# Cleaning text

In [3]:
def clean_text(text):
    text = (
        text.lower()
    )  # NOTE: talvez seja interessante deixar alguns caracteres em maiúsculo

    # remove URLs
    text = re.sub(r"http\\S+|www\\S+", " ", text)

    # keeps letters and spaces
    text = re.sub(
        r"[^a-zà-úâêîôûãõç\\s]", " ", text
    )  # NOTE: está tirando os números também

    # removes extra spaces
    text = re.sub(r"\\s+", " ", text).strip()

    return text

In [4]:
print(clean_text(text_test))

ótimo custo benefício  não tampa tão bem o sol  mas pelo valor é ok  enteguega o que propõe


# Features

In [ ]:
# 1 Number of ADJ
# 3 Number of ADV
# 6 Number of DET 
# 7 Number of INTJ
# 9 Number of NUM
# 12 Number of PUNCT
# 14 Number of SYM
# 16 Number of COMP
# 17 Number of SUP
# 18 Number of X
#
# 19 ADJ → NOUN
# 23 ADV → VERB + {PCP | GER | PS | IMPF}
#
# 24 Number of subjecticve terms
# 25 Number of positve terms
# 26 Number of negative terms
# 27 Number of marks
# 28 Proportion of subjective terms
# 29 Proportion of positive terms
# 30 Proportion of negative terms
# 31 Proportion of mark terms
#
# tabela 4 ignora
#
#
# 46 Ratio of the uppercase characters to the sentence length
#
# tabela 6 twitter
# 47 Number of URL
# 48 Number of mentions (@user)
# 49 Number of elongated words
# 50 Polarity of emojis
# 51 Polarity of emoticons

## Part-Of-Speech (POS) features

Don't forget to `python -m spacy download pt_core_news_sm`
See https://universaldependencies.org/u/pos/ for information on Tags

In [5]:
nlp = spacy.load("pt_core_news_sm")

In [6]:
# default text
doc_test = nlp(text_test)

In [19]:
# Feature 1
def number_of_adj(doc: Doc):
    logging.debug("[number_of_adj]")

    adj_num = 0
    for token in doc:
        if token.pos_ == "ADJ":
            adj_num += 1

            logging.debug(token.text + "\t|" + token.pos_)
    return adj_num


test = nlp("Que homem bonito")
print(test, "\nNumber of adj:", number_of_adj(test))

DEBUG:root:[number_of_adj]
DEBUG:root:bonito	|ADJ


Que homem bonito 
Number of adj: 1


In [22]:
# Feature 3
def number_of_adv(doc: Doc):
    adv_num = 0

    for token in doc:
        if token.pos_ == "ADV":
            adv_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return adv_num


test = nlp("Ele é muito bonito")
print(test, "\nNumber of adv:", number_of_adv(test))

DEBUG:root:muito	|ADV


Ele é muito bonito 
Number of adv: 1


In [23]:
# Feature 6
# NOTE: eu acho que essa feature é meio nada a ver -vini
def number_of_det(doc: Doc):
    det_num = 0

    for token in doc:
        if token.pos_ == "DET":
            det_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return det_num


print(doc_test, "\nNumber of det:", number_of_det(doc_test))

DEBUG:root:o	|DET


Ótimo custo benefício, não tampa tão bem o sol, mas pelo valor é ok. Enteguega o que propõe. 
Number of det: 1


In [24]:
# Feature 7
def number_of_intj(doc: Doc):
    intj_num = 0

    for token in doc:
        if token.pos_ == "INTJ":
            intj_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return intj_num


test = nlp("Ah, que droga!")
print(test, "\nNumber of intj:", number_of_intj(test))

DEBUG:root:Ah	|INTJ


Ah, que droga! 
Number of intj: 1


In [25]:
# Feature 9
def number_of_num(doc: Doc):
    num_num = 0

    for token in doc:
        if token.pos_ == "NUM":
            num_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return num_num


test = nlp("Dois pesos, 2 medidas")
print(test, "\nNumber of num:", number_of_num(test))

DEBUG:root:Dois	|NUM
DEBUG:root:2	|NUM


Dois pesos, 2 medidas 
Number of num: 2


In [26]:
# Feature 12
def number_of_punct(doc: Doc):
    punct_num = 0

    for token in doc:
        if token.pos_ == "PUNCT":
            punct_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return punct_num


test = nlp("Ah, que droga!")
print(test, "\nNumber of punct:", number_of_punct(test))

DEBUG:root:,	|PUNCT
DEBUG:root:!	|PUNCT


Ah, que droga! 
Number of punct: 2


In [27]:
# Feature 14
def number_of_sym(doc: Doc):
    sym_num = 0

    for token in doc:
        if token.pos_ == "SYM":
            sym_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return sym_num


test = nlp("R$50,00")
print(test, "\nNumber of sym:", number_of_sym(test))

DEBUG:root:R$	|SYM


R$50,00 
Number of sym: 1


In [42]:
# Feature 16 & 18
comp_sup_list = ["melhor", "pior", "maior", "menor"]


def number_of_comp_sup(text: str):
    """Comparative or superlative words"""
    text = text.lower()
    comp_sub_count = 0

    for word in comp_sup_list:
        comp_sub_count += text.count(word)

    return comp_sub_count


test = "A maior e melhor torta é a de limão, e a pior? Não sei"
print(number_of_comp_sup(test))

3


In [9]:
# Feature 18
# NOTE: This feature doesn't seem that meaningful -vini
# NOTE: Couldn't test it
def number_of_x(doc: Doc):
    """X means other"""
    x_num = 0

    for token in doc:
        if token.pos_ == "X":
            x_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return x_num


test = nlp("E aí ele xfgh pdl jklw sombrero fastfood meaningful")
print(test, "\nNumber of x:", number_of_x(test))

E aí ele xfgh pdl jklw sombrero fastfood meaningful 
Number of x: 0


## Syntactic Features

## Sentiment Features

In [ ]:
# provável que a gente tenha que fazer nossas próprias listas de palavras positivas e negativas

# Listas de palavras para features de sentimento (Features 25, 26, 28, 29)
# Baseadas em léxicos de sentimento para português
# Adaptadas ao domínio de reviews de produtos

POSITIVE_TERMS = [
    # Adjetivos positivos
    "bom", "boa", "bons", "boas",
    "ótimo", "ótima", "ótimos", "ótimas",
    "excelente", "excelentes",
    "maravilhoso", "maravilhosa", "maravilhosos", "maravilhosas",
    "perfeito", "perfeita", "perfeitos", "perfeitas",
    "incrível", "incríveis",
    "fantástico", "fantástica", "fantásticos", "fantásticas",
    "sensacional", "sensacionais",
    "lindo", "linda", "lindos", "lindas",
    "bonito", "bonita", "bonitos", "bonitas",
    "agradável", "agradáveis",
    "delicioso", "deliciosa", "deliciosos", "deliciosas",
    "gostoso", "gostosa", "gostosos", "gostosas",
    "rápido", "rápida", "rápidos", "rápidas",
    "eficiente", "eficientes",
    "resistente", "resistentes",
    "durável", "duráveis",
    "confortável", "confortáveis",
    "prático", "prática", "práticos", "práticas",
    "funcional", "funcionais",
    "satisfeito", "satisfeita", "satisfeitos", "satisfeitas",
    "feliz", "felizes",
    "top", "show", "incrível",
    # Verbos/expressões positivas
    "adorei", "amei", "gostei", "aprovei", "recomendo",
    "superou", "surpreendeu", "encantou",
    "funcionou", "funcionando", "chegou",
    "vale", "valeu", "valendo",
    # Substantivos positivos
    "qualidade", "excelência", "perfeição",
    "satisfação", "prazer", "amor",
]

NEGATIVE_TERMS = [
    # Adjetivos negativos
    "ruim", "ruins",
    "péssimo", "péssima", "péssimos", "péssimas",
    "horrível", "horríveis",
    "terrível", "terríveis",
    "lixo", "lixos",
    "fraco", "fraca", "fracos", "fracas",
    "quebrado", "quebrada", "quebrados", "quebradas",
    "defeituoso", "defeituosa", "defeituosos", "defeituosas",
    "demorado", "demorada", "demorados", "demoradas",
    "caro", "cara", "caros", "caras",
    "decepcionante", "decepcionantes",
    "insatisfatório", "insatisfatória",
    "inadequado", "inadequada", "inadequados", "inadequadas",
    "falso", "falsa", "falsos", "falsas",
    "inútil", "inúteis",
    "difícil", "difíceis",
    "desconfortável", "desconfortáveis",
    # Verbos/expressões negativos
    "odiei", "detestei", "decepcionou", "frustrou",
    "quebrou", "parou", "travou", "bugou",
    "arrependido", "arrependida",
    "devolvendo", "devolvi", "cancelei",
    # Substantivos negativos
    "decepção", "vergonha", "fraude", "lixo",
    "problema", "problemas", "defeito", "defeitos",
    "absurdo", "absurdos",
]

# Termos subjetivos = positivos + negativos + marcadores de opinião
OPINION_MARKERS = [
    "acho", "acredito", "penso", "considero", "creio",
    "parece", "parecer", "pareceu",
    "sinto", "senti", "sentindo",
    "esperava", "esperando",
    "recomendo", "recomendaria", "recomendaria",
    "vale", "valeu", "vale a pena",
]

SUBJECTIVE_TERMS = set(POSITIVE_TERMS + NEGATIVE_TERMS + OPINION_MARKERS)

# Pré-computar sets para busca O(1)
POSITIVE_SET  = set(POSITIVE_TERMS)
NEGATIVE_SET  = set(NEGATIVE_TERMS)

print(f"Positive terms: {len(POSITIVE_SET)}")
print(f"Negative terms: {len(NEGATIVE_SET)}")
print(f"Subjective terms (total): {len(SUBJECTIVE_TERMS)}")


'# provável que a gente tenha que fazer nossas próprias listas de palavras positivas e negativas\n\n# Listas de palavras para features de sentimento (Features 25, 26, 28, 29)\n# Baseadas em léxicos de sentimento para português\n# Adaptadas ao domínio de reviews de produtos\n\nPOSITIVE_TERMS = [\n    # Adjetivos positivos\n    "bom", "boa", "bons", "boas",\n    "ótimo", "ótima", "ótimos", "ótimas",\n    "excelente", "excelentes",\n    "maravilhoso", "maravilhosa", "maravilhosos", "maravilhosas",\n    "perfeito", "perfeita", "perfeitos", "perfeitas",\n    "incrível", "incríveis",\n    "fantástico", "fantástica", "fantásticos", "fantásticas",\n    "sensacional", "sensacionais",\n    "lindo", "linda", "lindos", "lindas",\n    "bonito", "bonita", "bonitos", "bonitas",\n    "agradável", "agradáveis",\n    "delicioso", "deliciosa", "deliciosos", "deliciosas",\n    "gostoso", "gostosa", "gostosos", "gostosas",\n    "rápido", "rápida", "rápidos", "rápidas",\n    "eficiente", "eficientes",\n    

In [ ]:
'''# carrega uma vez
op = pd.read_csv("lexico_v3.0.txt", sep=",", header=None,
                 names=["word", "pos", "polarity", "type"])

POSITIVE_SET = set(op[op["polarity"] ==  1]["word"])
NEGATIVE_SET = set(op[op["polarity"] == -1]["word"])

# as funções do notebook continuam idênticas
def number_of_positive_terms(doc):
    return sum(1 for t in doc 
               if t.is_alpha and t.lemma_.lower() in POSITIVE_SET)
'''

In [46]:
# Feature 25 — Number of positive terms
def number_of_positive_terms(doc: Doc):
    """
    Conta tokens cujo lema (em minúsculas) está na lista de termos positivos.
    Usa o lema para capturar flexões: 'excelentes' -> 'excelente'.
    Exemplo: 'As comidas são excelentes.' -> 1
    """
    count = 0
    for token in doc:
        if token.is_alpha and token.lemma_.lower() in POSITIVE_SET:
            count += 1
            logging.debug(f"[positive] {token.text} -> {token.lemma_}")
    return count


test_pos = nlp("A maior e melhor torta é a de limão, e a pior? Não sei")
test_neg = nlp("Atendimento da recepcionista da porta ruim.")
print("'As comidas são excelentes.':", number_of_positive_terms(test_pos))  # esperado: 1
print("'Atendimento da recepcionista da porta ruim.':", number_of_positive_terms(test_neg))  # esperado: 0

print(number_of_positive_terms(test_pos))

DEBUG:root:[positive] melhor -> melhor
DEBUG:root:[positive] é -> ser


DEBUG:root:[positive] melhor -> melhor
DEBUG:root:[positive] é -> ser


'As comidas são excelentes.': 2
'Atendimento da recepcionista da porta ruim.': 0
2


In [ ]:
# Feature 26 — Number of negative terms
def number_of_negative_terms(doc: Doc):
    """
    Conta tokens cujo lema (em minúsculas) está na lista de termos negativos.
    Usa o lema para capturar flexões: 'ruins' -> 'ruim'.
    Exemplo: 'Atendimento da recepcionista da porta ruim.' -> 1
    """
    count = 0
    for token in doc:
        if token.is_alpha and token.lemma_.lower() in NEGATIVE_SET:
            count += 1
            logging.debug(f"[negative] {token.text} -> {token.lemma_}")
    return count


test_pos = nlp("As comidas são excelentes.")
test_neg = nlp("Atendimento da recepcionista da porta ruim.")
print("'As comidas são excelentes.' :", number_of_negative_terms(test_pos))   # esperado: 0
print("'Atendimento da recepcionista da porta ruim.':", number_of_negative_terms(test_neg))  # esperado: 1


DEBUG:root:[negative] comidas -> comida
DEBUG:root:[negative] ruim -> ruim


'As comidas são excelentes.' : 1
'Atendimento da recepcionista da porta ruim.': 1


In [ ]:
# Feature 28 — Proportion of subjective terms
def proportion_of_subjective_terms(doc: Doc):
    """
    Proporção de tokens alfanuméricos que são termos subjetivos
    (positivos + negativos + marcadores de opinião).
    Retorna valor entre 0 e 1, arredondado em 3 casaas.
    Exemplo: 'Comida boa por um bom preço.' -> 0.429 (2 subjetivos de 7 tokens alpha)
    """
    alpha_tokens = [t for t in doc if t.is_alpha]
    if not alpha_tokens:
        return 0.0

    subjective_count = sum(
        1 for t in alpha_tokens
        if t.lemma_.lower() in SUBJECTIVE_TERMS
    )
    return round(subjective_count / len(alpha_tokens), 3)


test_sub = nlp("Comida boa por um bom preço.")
print("'Comida boa por um bom preço.':", proportion_of_subjective_terms(test_sub))  # esperado: ~0.429


'Comida boa por um bom preço.' -> 0.333


In [29]:
# Feature 29 — Proportion of positive terms
def proportion_of_positive_terms(doc: Doc):
    """
    Proporção de tokens alfanuméricos que são termos positivos.
    Retorna valor entre 0 e 1, aredondado em 3 casas.
    Exemplo: 'As comidas são excelente s.' -> 0.2 (1 positivo de 5 tokens alpha)
    """
    alpha_tokens = [t for t in doc if t.is_alpha]
    if not alpha_tokens:
        return 0.0

    positive_count = sum(
        1 for t in alpha_tokens
        if t.lemma_.lower() in POSITIVE_SET
    )
    return round(positive_count / len(alpha_tokens), 3)


test_pos = nlp("As comidas são excelentes.")
print(test_pos)
print("'As comidas são excelentes.':", proportion_of_positive_terms(test_pos))  # esperado: 0.2


As comidas são excelentes.
'As comidas são excelentes.': 0.25


## Structural Features

In [46]:
def uppercase_ratio(text: str):
    # erases spaces
    re.sub("\\s", "", text)

    length = len(text)

    upper_count = len(re.findall("[A-Z]", text))

    return round(upper_count / length, 3)


test1 = "Eu absolutamente ADORO GIRAFFAS"
test2 = "sei lá, tanto faz. Sabe?"

print("test 1: ", uppercase_ratio(test1))
print("test 2: ", uppercase_ratio(test2))

test 1:  0.452
test 2:  0.042


## Twitter Features


In [18]:
# Feature 47 — Number of URLs
def number_of_urls(text: str):
    """
    Conta URLs no texto usando regex.
    Exemplo: 'Vendo! Compartilha aí gente! http://fb.me/24tl16IwA' -> 1
    """
    url_pattern = re.compile(
        r"http\S+|www\.\S+",
        re.IGNORECASE
    )
    return len(url_pattern.findall(text))


test1 = "Vendo! Compartilha aí gente! http://fb.me/24tl16IwA"
test2 = "Produto sem link nenhum"
print(repr(test1), ":", number_of_urls(test1))  # esperado: 1
print(repr(test2), ":", number_of_urls(test2))  # esperado: 0

'Vendo! Compartilha aí gente! http://fb.me/24tl16IwA' : 1
'Produto sem link nenhum' : 0


In [ ]:
# Feature 48 — Number of mentions (@user)
def number_of_mentions(text: str):
    """
    Conta menções no formato @usuario.
    Exemplo: 'Comprei um notebook da @Dell vamos ver se eu vou gostar!!!' -> 1
    """
    mention_pattern = re.compile(r"@\w+")
    return len(mention_pattern.findall(text))


test1 = "Comprei um notebook da @Dell vamos ver se eu vou gostar!!!"
test2 = "Produto sem menção nenhuma"
print(repr(test1), ":", number_of_mentions(test1))  # esperado: 1
print(repr(test2), ":", number_of_mentions(test2))  # esperado: 0


In [ ]:
# Feature 49 — Number of elongated words
def number_of_elongated_words(text: str):
    """
    Conta palavras com letras repetidas 3+ vezes consecutivas.
    Esse padrão indica ênfase emocional em reviews informais.
    Exemplo: 'Essa Dell é demais... Note chegoooou?' -> 1  ('chegoooou')
    Exemplo: 'Note arriiiived?' -> 1  ('arriiiived')
    """
    elongated_pattern = re.compile(r"\b\w*(.)\1{2,}\w*\b")
    return len(elongated_pattern.findall(text))


test1 = "Essa Dell é demais... Note chegoooou?"
test2 = "Produto normal sem ênfase"
test3 = "arriiiived e chegoooou ambos elongados"
print(repr(test1), ":", number_of_elongated_words(test1))  # esperado: 1
print(repr(test2), ":", number_of_elongated_words(test2))  # esperado: 0
print(repr(test3), ":", number_of_elongated_words(test3))  # esperado: 2


In [ ]:
# Feature 50 — Polarity of emojis
# Subconjunto de emojis comuns em reviews de produtos em português
EMOJI_POLARITY = {
    # =========================
    # POSITIVOS (+1)
    # =========================
    
    # Carinhas felizes
    "😀": 1, "😃": 1, "😄": 1, "😁": 1, "😆": 1, "😅": 1,
    "😂": 1, "🤣": 1, "😊": 1, "🙂": 1, "🙃": 1,
    "😉": 1, "😌": 1, "😇": 1, "😋": 1,
    
    # Amor / carinho
    "😍": 1, "🥰": 1, "😘": 1, "😗": 1, "😙": 1, "😚": 1,
    "💋": 1, "❤️": 1, "🧡": 1, "💛": 1, "💚": 1, "💙": 1, "💜": 1,
    "🤍": 1, "🤎": 1, "💕": 1, "💞": 1, "💓": 1, "💗": 1, "💖": 1,
    "💘": 1, "💝": 1,
    
    # Confiança / atitude
    "😎": 1, "🤩": 1, "🥳": 1, "😏": 1,
    "💪": 1, "🔥": 1, "🚀": 1,
    
    # Aprovação / gestos
    "👍": 1, "👌": 1, "👏": 1, "🙌": 1, "🤝": 1,
    "✌️": 1, "🤙": 1, "🫶": 1,
    
    # Sucesso / celebração
    "🎉": 1, "🎊": 1, "🎁": 1, "🏆": 1, "🥇": 1, "🥈": 1, "🥉": 1,
    "🌟": 1, "⭐": 1, "✨": 1, "💫": 1,
    
    # Check / positivo
    "✅": 1, "☑️": 1, "✔️": 1,
    
    # Comida boa (geralmente positivo)
    "🍕": 1, "🍔": 1, "🍟": 1, "🍩": 1, "🍰": 1, "🍫": 1,
    "🍻": 1, "🥂": 1,
    
    # =========================
    # NEGATIVOS (-1)
    # =========================
    
    # Tristeza
    "😢": -1, "😭": -1, "😿": -1,
    "😞": -1, "😟": -1, "😔": -1, "😕": -1,
    "🙁": -1, "☹️": -1,
    
    # Raiva
    "😡": -1, "😠": -1, "🤬": -1, "😤": -1,
    "👿": -1,
    
    # Frustração / cansaço
    "😣": -1, "😖": -1, "😩": -1, "😫": -1,
    "🥱": -1, "😪": -1,
    
    # Medo / ansiedade
    "😱": -1, "😨": -1, "😰": -1, "😥": -1,
    "😓": -1,
    
    # Doença / mal-estar
    "🤒": -1, "🤕": -1, "🤢": -1, "🤮": -1,
    
    # Desaprovação
    "👎": -1, "🙅": -1, "🙅‍♂️": -1, "🙅‍♀️": -1,
    "🚫": -1, "⛔": -1, "❌": -1,
    
    # Decepção / vergonha
    "🤦": -1, "🤦‍♂️": -1, "🤦‍♀️": -1,
    "😳": -1, "🥴": -1,
    
    # Coração negativo
    "💔": -1, "🖤": -1,
    
    # Coisas ruins / lixo
    "🗑️": -1, "💩": -1,
    
    # Perigo / alerta
    "⚠️": -1, "🚨": -1,
}


def polarity_of_emojis(text: str):
    """
    Calcula a polaridade média dos emojis presentes no texto.
    Retorna valor entre -1 e 1 (ou 0 se não houver emojis reconhecidos).
    Exemplo: 'O cara da dell vem amanha arrumar meu not' -> 0.693
    """
    scores = [
        EMOJI_POLARITY[char]
        for char in text
        if char in EMOJI_POLARITY
    ]
    if not scores:
        return 0
    return round(sum(scores) / len(scores), 3)


test1 = "Produto chegou perfeito! 😍👍✨"
test2 = "Horrível, quebrou em 2 dias. 👎😭"
test3 = "Produto ok, sem emoji"
print(repr(test1), ":", polarity_of_emojis(test1))  # esperado: 1.0
print(repr(test2), ":", polarity_of_emojis(test2))  # esperado: -1.0
print(repr(test3), ":", polarity_of_emojis(test3))  # esperado: 0


'Produto chegou perfeito! 😍👍✨👎😭' : 0.2
'Horrível, quebrou em 2 dias. 👎😭' : -1.0
'Produto ok, sem emoji' : 0


In [25]:
# Feature 51 — Polarity of emoticons
# Emoticons são sequências de caracteres ASCII (ex: :-) :-( )
EMOTICON_POLARITY = {
    # =========================
    # POSITIVOS (+1)
    # =========================
    
    # Sorrisos básicos
    ":)": 1, ":-)": 1, ":]": 1, ":-]": 1, "=]": 1, "=)": 1,
    ":}": 1, ":-}": 1,
    
    # Risadas
    ":D": 1, ":-D": 1, "=D": 1, "xD": 1, "XD": 1, "X-D": 1,
    "x-D": 1,
    
    # Piscadas
    ";)": 1, ";-)": 1, "*-)": 1, "*)": 1,
    
    # Língua / zoeira
    ":P": 1, ":-P": 1, ":p": 1, ":-p": 1,
    "=P": 1, "=p": 1, "XP": 1, "xp": 1,
    
    # Amor / carinho
    "<3": 1, "</3": -1,  # (já colocando o oposto aqui)
    ":*": 1, ":-*": 1,
    
    # Anime / kawaii
    "^^": 1, "^_^": 1, "^-^": 1, "^.^": 1,
    "(^_^)": 1, "(^.^)": 1, "(^-^)": 1,
    "(^o^)": 1,
    
    # Outros positivos
    ":')": 1,  # choro de felicidade
    ":>": 1, ":-)": 1,
    "(:": 1, "(-:": 1,
    
    # =========================
    # NEGATIVOS (-1)
    # =========================
    
    # Tristeza
    ":(": -1, ":-(": -1, ":[": -1, ":-[": -1,
    ":{": -1, ":-{": -1,
    "=(":-1 if False else -1,  # evita duplicação visual bug (mantém válido)
    "=(": -1,
    
    # Choro
    ":'(": -1, ":'-(": -1,
    "T_T": -1, "TT_TT": -1, ";_;": -1,
    "(T_T)": -1,
    
    # Decepção / neutro negativo
    ":/": -1, ":-/": -1, ":\\": -1, ":-\\": -1,
    ":|": -1, ":-|": -1,
    ">:|": -1,
    
    # Raiva
    ">:(": -1, ">:-(": -1,
    "D:<": -1, "D-:<": -1,
    
    # Choque / medo
    "D:": -1, "D-:": -1,
    
    # Vergonha / desconforto
    ":$": -1, ":-$": -1,
    ":-X": -1, ":X": -1,
    
    # Confusão
    "o_O": -1, "O_o": -1, "o.O": -1, "O.O": -1,
    
    # Cansaço
    "-_-": -1, "-__-": -1,
    
    # Outros negativos
    "):": -1, ")-:": -1,
}

# Regex para detectar emoticons — ordem importa: mais longos primeiro
_emoticon_pattern = re.compile(
    r"|" .join(re.escape(e) for e in sorted(EMOTICON_POLARITY, key=len, reverse=True))
)


def polarity_of_emoticons(text: str):
    """
    Calcula a polaridade média dos emoticons ASCII presentes no texto.
    Retorna -1, 0 ou 1 (ou média se houver mistura).
    Exemplo: 'Notebook #dell sem teclado #abnt2 desanima... :-(' -> -1
    """
    matches = _emoticon_pattern.findall(text)
    scores = [EMOTICON_POLARITY[m] for m in matches if m in EMOTICON_POLARITY]
    if not scores:
        return 0
    return round(sum(scores) / len(scores), 3)


test1 = "Notebook #dell sem teclado #abnt2 desanima... :-("
test2 = "Chegou antes do prazo!! :D :-)"
test3 = "Produto comum, sem emoticon"
print(repr(test1), "->", polarity_of_emoticons(test1))  # esperado: -1
print(repr(test2), "->", polarity_of_emoticons(test2))  # esperado: 1.0
print(repr(test3), "->", polarity_of_emoticons(test3))  # esperado: 0


'Notebook #dell sem teclado #abnt2 desanima... :-(' -> -1.0
'Chegou antes do prazo!! :D :-)' -> 1.0
'Produto comum, sem emoticon' -> 0


### spaCy testing

In [27]:
doc = nlp(text_test)
for token in doc:
    print(token.text, "\t\t|", token.pos_, token.dep_)

Ótimo 		| PROPN nsubj
custo 		| VERB nsubj
benefício 		| ADJ obj
, 		| PUNCT punct
não 		| ADV advmod
tampa 		| VERB parataxis
tão 		| ADV advmod
bem 		| ADV advmod
o 		| DET det
sol 		| NOUN nsubj
, 		| PUNCT punct
mas 		| CCONJ cc
pelo 		| ADP case
valor 		| NOUN conj
é 		| AUX cop
ok 		| NOUN ROOT
. 		| PUNCT punct
Enteguega 		| VERB ROOT
o 		| PRON obj
que 		| PRON nsubj
propõe 		| VERB acl:relcl
. 		| PUNCT punct


In [28]:
displacy.render(doc, style="dep", options={"compact": "True"})